# 05. Fit an international logistic model and blend

The global model uses global Elo difference. The international model learns separate coefficients for local Elo difference and pool-offset difference, instead of forcing those strengths to contribute equally. Apply it only to MSI, WLDs, and FST with available regional features.

Run notebooks 01-04 first. Reuse notebook 03's exact boundaries, select a blend using validation only, then refit and evaluate with the choice fixed. Outside international scope or when regional inputs are unavailable, use global logistic regression. The already-inspected test period remains development evidence; later final claims require an untouched period.


## 01. Load and align chronological inputs

Read global and calibrated regional exports, require matching identities and outcomes, and merge one-to-one. Derive pool-offset difference from the two prior offsets. Regional features are available only for modeling-scope games with known homes and finite inputs. Missing regional information remains missing rather than being imputed as zero.


In [1]:
from pathlib import Path
import json
import hashlib
import joblib
import numpy as np
import pandas as pd
import sklearn
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, log_loss, brier_score_loss, roc_auc_score

base = Path.cwd()
if not (base / "data" / "processed").is_dir():
    base = base.parent
processed = base / "data" / "processed"
regional_path = processed / "calibrated_elo_features_all_years.csv"
global_path = processed / "elo_features_all_years.csv"
regional = pd.read_csv(regional_path)
global_features = pd.read_csv(global_path)
for frame in [regional, global_features]:
    frame["date"] = pd.to_datetime(frame["date"], utc=True)
    assert frame["match_id"].is_unique
identity = ["match_id", "date", "league", "team_a", "team_b", "result_a"]
pd.testing.assert_frame_equal(
    regional[identity].sort_values("match_id").reset_index(drop=True),
    global_features[identity].sort_values("match_id").reset_index(drop=True),
)
data = regional.merge(global_features[["match_id", "elo_diff", "elo_probability_a"]],
                      on="match_id", validate="one_to_one").sort_values(["date", "match_id"]).reset_index(drop=True)
data["pool_offset_diff"] = data["pre_pool_offset_a"] - data["pre_pool_offset_b"]
GLOBAL_FEATURES = ["elo_diff"]
REGIONAL_FEATURES = ["local_elo_diff", "pool_offset_diff"]
assert data["use_international_model"].isin([True, False]).all()
data["international_scope"] = data["use_international_model"]
data["regional_available"] = (
    data["international_scope"] & data["home_known_a"] & data["home_known_b"]
    & np.isfinite(data[REGIONAL_FEATURES]).all(axis=1)
)
print("Total games:", len(data))
print("International games with regional features:", int(data["regional_available"].sum()))

Total games: 96075
International games with regional features: 1860


## 02. Reuse saved timestamp boundaries

Read notebook 03's boundaries and require identical period counts and date limits. Keep equal timestamps together and verify strict separation. The split is reused rather than selected again from model performance.


In [2]:
split_path = base / "results" / "baseline" / "chronological_splits.csv"
splits = pd.read_csv(split_path).set_index("split")
validation_start = pd.Timestamp(splits.loc["validation", "start"])
test_start = pd.Timestamp(splits.loc["testing", "start"])
train = data.loc[data["date"].lt(validation_start)].copy()
validation = data.loc[data["date"].ge(validation_start) & data["date"].lt(test_start)].copy()
test = data.loc[data["date"].ge(test_start)].copy()
for name, frame in [("training", train), ("validation", validation), ("testing", test)]:
    assert len(frame) == int(splits.loc[name, "matches"])
    assert frame["date"].min() == pd.Timestamp(splits.loc[name, "start"])
    assert frame["date"].max() == pd.Timestamp(splits.loc[name, "end"])
assert train["date"].max() < validation["date"].min()
assert validation["date"].max() < test["date"].min()
print("Training / validation / test:", len(train), len(validation), len(test))

Training / validation / test: 57656 19215 19204


## 03. Fit symmetric models on the first period

Train the global pipeline on all training games and the regional pipeline on available international training games. Both fit a scaler on mirrored training examples and use C=1 with no intercept. The regional probability is `sigmoid(beta_local * scaled_local_diff + beta_pool * scaled_pool_diff)`. Check opponent symmetry and predict validation with fixed coefficients, explicitly falling back to global logistic when regional prediction is unavailable.


In [3]:
def fit_mirrored(frame, columns):
    x = frame[columns].to_numpy(float)
    y = frame["result_a"].to_numpy(int)
    if len(y) == 0 or len(np.unique(y)) != 2 or not np.isfinite(x).all():
        raise ValueError("Training requires finite features and both outcomes.")
    model = make_pipeline(StandardScaler(with_mean=False),
                          LogisticRegression(C=1.0, fit_intercept=False, max_iter=1000))
    model.fit(np.concatenate([x, -x]), np.concatenate([y, 1 - y]))
    sample = x[:min(50, len(x))]
    np.testing.assert_allclose(model.predict_proba(sample)[:, 1] + model.predict_proba(-sample)[:, 1], 1)
    return model

def predict_pair(frame, global_model, regional_model):
    output = frame.copy()
    output["global_logistic_probability_a"] = global_model.predict_proba(output[GLOBAL_FEATURES].to_numpy(float))[:, 1]
    available = output["regional_available"]
    output["regional_logistic_probability_a"] = np.nan
    if available.any():
        output.loc[available, "regional_logistic_probability_a"] = regional_model.predict_proba(output.loc[available, REGIONAL_FEATURES].to_numpy(float))[:, 1]
    output["regional_with_global_fallback"] = output["regional_logistic_probability_a"].fillna(output["global_logistic_probability_a"])
    assert output["regional_with_global_fallback"].between(0, 1).all()
    return output

validation_global_model = fit_mirrored(train, GLOBAL_FEATURES)
validation_regional_model = fit_mirrored(train.loc[train["regional_available"]], REGIONAL_FEATURES)
validation_predictions = predict_pair(validation, validation_global_model, validation_regional_model)
print("First-period regional training games:", int(train["regional_available"].sum()))

First-period regional training games: 1047


## 04. Select the blend using validation only

Try the predeclared regional weights 0, 0.1, ..., 1 with `p_blend = (1 - weight) * p_global + weight * p_regional`. Score every international validation game, including global fallback when needed. Choose minimum log loss, breaking ties toward the smaller regional weight. The table records the full search; test scores do not select the weight.


In [4]:
BLEND_WEIGHTS = np.linspace(0.0, 1.0, 11)
validation_international = validation_predictions.loc[validation_predictions["international_scope"]]
if validation_international.empty:
    raise ValueError("No international validation matches.")
blend_rows = []
for weight in BLEND_WEIGHTS:
    probability = ((1 - weight) * validation_international["global_logistic_probability_a"]
                   + weight * validation_international["regional_with_global_fallback"])
    blend_rows.append({"regional_weight": float(weight), "matches": len(validation_international),
                       "log_loss": log_loss(validation_international["result_a"], probability, labels=[0, 1]),
                       "brier": brier_score_loss(validation_international["result_a"], probability)})
blend_grid = pd.DataFrame(blend_rows)
selected_weight = float(blend_grid.sort_values(["log_loss", "regional_weight"]).iloc[0]["regional_weight"])
print("Validation-selected regional weight:", selected_weight)
print("Validation regional availability:", int(validation_international["regional_available"].sum()), "/", len(validation_international))
display(blend_grid)

Validation-selected regional weight: 0.8
Validation regional availability: 423 / 423


,regional_weight,matches,log_loss,brier
0,0.0,423,0.688159,0.246717
1,0.1,423,0.671378,0.239352
2,0.2,423,0.657438,0.232924
3,0.3,423,0.645808,0.227435
4,0.4,423,0.636195,0.222883
5,0.5,423,0.628444,0.219269
6,0.6,423,0.622502,0.216593
7,0.7,423,0.618411,0.214854
8,0.8,423,0.616333,0.214054
9,0.9,423,0.616628,0.214191


## 05. Refit development models and freeze the choice

Refit both pipelines using training plus validation while keeping features, C, and blend weight unchanged. Require the global refit to reproduce notebook 03's saved probabilities and unavailable regional inputs to produce the global probability. Coefficients remain fixed through test, while upstream Elo features continue using earlier results. This is online evaluation rather than a tournament-start forecast.


In [5]:
development = data.loc[data["date"].lt(test_start)].copy()
assert development["date"].max() < test["date"].min()
final_global_model = fit_mirrored(development, GLOBAL_FEATURES)
final_regional_model = fit_mirrored(development.loc[development["regional_available"]], REGIONAL_FEATURES)
test_predictions = predict_pair(test, final_global_model, final_regional_model)
baseline_predictions = pd.read_csv(base / "results" / "baseline" / "test_predictions.csv")
assert set(test_predictions["match_id"]) == set(baseline_predictions["match_id"])
check = test_predictions[["match_id", "global_logistic_probability_a"]].merge(
    baseline_predictions[["match_id", "logistic_probability_a"]], on="match_id", validate="one_to_one")
np.testing.assert_allclose(check["global_logistic_probability_a"], check["logistic_probability_a"])
for frame in [validation_predictions, test_predictions]:
    frame["blend_probability_a"] = ((1 - selected_weight) * frame["global_logistic_probability_a"]
                                    + selected_weight * frame["regional_with_global_fallback"])
    unavailable = ~frame["regional_available"]
    np.testing.assert_allclose(frame.loc[unavailable, "blend_probability_a"],
                               frame.loc[unavailable, "global_logistic_probability_a"])
    assert frame["blend_probability_a"].between(0, 1).all()
print("Final regional training games:", int(development["regional_available"].sum()))
print("Baseline global-logistic reproduction passed.")

Final regional training games: 1470
Baseline global-logistic reproduction passed.


## 06. Report metrics, coverage, and calibration

Report validation and test for all games, international scope, MSI plus WLDs, and WLDs. Compare global Elo, global logistic, regional logistic with global fallback, and the selected blend on full scope and the common available subset. Accuracy and AUC are higher-is-better; log loss and Brier are lower-is-better. Read coverage and counts alongside scores and inspect confidence with calibration bins. Domestic and other out-of-scope games intentionally use global logistic.


In [6]:
def report_predictions(predictions, period):
    masks = {
        "all": pd.Series(True, index=predictions.index),
        "international": predictions["international_scope"],
        "msi_and_worlds": predictions["league"].isin(["MSI", "WLDs"]),
        "worlds": predictions["league"].eq("WLDs"),
    }
    models = {"global_elo": "elo_probability_a", "global_logistic": "global_logistic_probability_a",
              "regional_logistic": "regional_with_global_fallback", "validation_selected_blend": "blend_probability_a"}
    rows, coverage, bins = [], [], []
    for scope, mask in masks.items():
        scoped = predictions.loc[mask]
        available = scoped["regional_available"]
        coverage.append({"period": period, "scope": scope, "matches": len(scoped),
                         "regional_available": int(available.sum()), "global_fallback_matches": int((~available).sum()),
                         "regional_coverage": available.mean() if len(scoped) else np.nan})
        for population, group in [("full_scope", scoped), ("common_available", scoped.loc[available])]:
            for model, column in models.items():
                y, p = group["result_a"], group[column]
                values = {"accuracy": np.nan, "log_loss": np.nan, "brier": np.nan, "roc_auc": np.nan}
                if len(group):
                    values = {"accuracy": accuracy_score(y, p >= .5), "log_loss": log_loss(y, p, labels=[0, 1]),
                              "brier": brier_score_loss(y, p),
                              "roc_auc": roc_auc_score(y, p) if y.nunique() == 2 else np.nan}
                rows.append({"period": period, "scope": scope, "population": population,
                             "model": model, "matches": len(group), **values})
                table = group.assign(probability_bin=pd.cut(p, np.linspace(0, 1, 11), include_lowest=True)).groupby(
                    "probability_bin", observed=False).agg(predicted_probability=(column, "mean"),
                    observed_win_rate=("result_a", "mean"), matches=("result_a", "size")).reset_index()
                assert table["matches"].sum() == len(group)
                bins.append(table.assign(period=period, scope=scope, population=population, model=model))
    return pd.DataFrame(rows), pd.DataFrame(coverage), pd.concat(bins, ignore_index=True)

validation_report, validation_coverage, validation_calibration = report_predictions(validation_predictions, "validation")
test_report, test_coverage, test_calibration = report_predictions(test_predictions, "test")
display(validation_report.loc[validation_report["scope"].eq("international") & validation_report["population"].eq("full_scope")])
display(test_coverage)
display(test_report.loc[test_report["population"].eq("full_scope")])

,period,scope,population,model,matches,accuracy,log_loss,brier,roc_auc
8,validation,international,full_scope,global_elo,423,0.546099,0.690800,0.247716,0.591648
9,validation,international,full_scope,global_logistic,423,0.546099,0.688159,0.246717,0.591648
10,validation,international,full_scope,regional_logistic,423,0.657210,0.620131,0.215266,0.715931
11,validation,international,full_scope,validation_selected_blend,423,0.657210,0.616333,0.214054,0.715517


,period,scope,matches,regional_available,global_fallback_matches,regional_coverage
0,test,all,19204,390,18814,0.020308
1,test,international,390,390,0,1.000000
2,test,msi_and_worlds,310,310,0,1.000000
3,test,worlds,159,159,0,1.000000


,period,scope,population,model,matches,accuracy,log_loss,brier,roc_auc
0,test,all,full_scope,global_elo,19204,0.630077,0.640541,0.224841,0.681242
1,test,all,full_scope,global_logistic,19204,0.630077,0.640172,0.224736,0.681242
2,test,all,full_scope,regional_logistic,19204,0.631171,0.640206,0.224670,0.682130
3,test,all,full_scope,validation_selected_blend,19204,0.631223,0.639801,0.224549,0.682237
8,test,international,full_scope,global_elo,390,0.610256,0.652837,0.231335,0.633202
9,test,international,full_scope,global_logistic,390,0.610256,0.652942,0.231273,0.633202
10,test,international,full_scope,regional_logistic,390,0.664103,0.654603,0.228009,0.691964
11,test,international,full_scope,validation_selected_blend,390,0.666667,0.634704,0.222051,0.695245
16,test,msi_and_worlds,full_scope,global_elo,310,0.590323,0.662907,0.236111,0.619237
17,test,msi_and_worlds,full_scope,global_logistic,310,0.590323,0.662637,0.235873,0.619237


## 07. Save models and the selection record

Save validation-stage and refitted models, selected weight, split boundaries, training counts, fixed settings, library version, and input hashes. Export the blend grid, predictions, metrics, coverage, and calibration under `results/international_model/`. This records the chosen approach without selecting it again from test results. Historical gold/objective features, roster modeling, periodic retraining, and official tournament simulation remain future work.


In [7]:
results_dir = base / "results" / "international_model"
results_dir.mkdir(parents=True, exist_ok=True)
model_dir = base / "models"
model_dir.mkdir(parents=True, exist_ok=True)
metadata = {
    "global_features": GLOBAL_FEATURES, "regional_features": REGIONAL_FEATURES,
    "validation_start": validation_start.isoformat(), "test_start": test_start.isoformat(),
    "trained_through": development["date"].max().isoformat(),
    "validation_regional_training_games": int(train["regional_available"].sum()),
    "final_regional_training_games": int(development["regional_available"].sum()),
    "final_global_training_games": len(development), "selected_regional_weight": selected_weight,
    "blend_weight_candidates": BLEND_WEIGHTS.tolist(),
    "selection_objective": "full-scope international validation log loss; ties prefer lower regional weight",
    "fixed_settings": {"C": 1.0, "fit_intercept": False, "mirrored_training": True, "scaler_with_mean": False},
    "scope": ["MSI", "WLDs", "FST"],
    "fallback": "global logistic outside international scope or when regional features are unavailable",
    "evaluation": "online Elo state, fixed logistic coefficients during each evaluation period",
    "test_status": "previously inspected development period; not untouched final evaluation",
    "limitations": ["league roles are not year-specific", "exact team names and potentially stale domestic homes",
                    "no roster features", "no frozen pre-tournament forecast"],
    "sklearn_version": sklearn.__version__,
    "input_sha256": {path.name: hashlib.sha256(path.read_bytes()).hexdigest()
                     for path in [regional_path, global_path, split_path]},
}
model_bundle = {
    "metadata": metadata, "validation_global_model": validation_global_model,
    "validation_regional_model": validation_regional_model,
    "global_model": final_global_model, "regional_model": final_regional_model,
}
joblib.dump(model_bundle, model_dir / "international_logistic_blend.joblib")
for name, frame in {
    "validation_blend_grid": blend_grid, "validation_metrics": validation_report,
    "test_metrics": test_report, "validation_coverage": validation_coverage,
    "test_coverage": test_coverage, "validation_calibration": validation_calibration,
    "test_calibration": test_calibration, "validation_predictions": validation_predictions,
    "test_predictions": test_predictions,
}.items():
    frame.to_csv(results_dir / f"{name}.csv", index=False)
(results_dir / "model_metadata.json").write_text(json.dumps(metadata, indent=2) + "\n", encoding="utf-8")
print("Saved international models and results:", results_dir)

Saved international models and results: C:\Users\USER\lol_worlds_predictor\results\international_model
